<a href="https://colab.research.google.com/github/davidekim/WRAPs/blob/main/helical_wraps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Pipeline example for creating helical WRAPs**

In [ ]:
#@title **Setup RFdiffusion** (~5-10min)
%%time
import os, time
import sys
import subprocess

def run_cmd(cmd):
  process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, shell=True, text=True)
  for line in iter(process.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()
  process.stdout.close()
  process.wait()

if not os.path.isdir("RFdiffusion"):
  print("installing RFdiffusion...")
  os.system("git clone https://github.com/RosettaCommons/RFdiffusion.git")
  # install dependencies
  os.system("pip install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
  os.system("pip install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
  os.system("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
  os.system("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
  os.system("cd RFdiffusion/env/SE3Transformer; pip install .")
  os.system("pip install biopython==1.81")
  os.system("pip install -U dm-haiku")
  os.system("pip install ml-collections")
  os.system('pip install --upgrade "jax[cuda12_pip]<0.6.0" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html')
  os.system("cd RFdiffusion; pip install -e .")
  os.system("pip install py3Dmol")
print()

os.environ["DGLBACKEND"] = "pytorch"
#os.environ["HYDRA_FULL_ERROR"] = '1'
diffusion_script = "RFdiffusion/scripts/run_inference.py"

In [ ]:
#@title **Install RFdiffusion weights** (~1-5min)

if not os.path.isdir("RFdiffusion/models"):
  print("installing RFdiffusion weights...")
  os.system("cd RFdiffusion; mkdir models && cd models")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/60f09a193fb5e5ccdc4980417708dbab/Complex_Fold_base_ckpt.pt")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/74f51cfb8b440f50d70878e05361d8f0/InpaintSeq_ckpt.pt")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/76d00716416567174cdb7ca96e208296/InpaintSeq_Fold_ckpt.pt")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/5532d2e1f3a4738decd58b19d633b3c3/ActiveSite_ckpt.pt")
  os.system("cd RFdiffusion/models; wget http://files.ipd.uw.edu/pub/RFdiffusion/12fc204edeae5b57713c5ad7dcb97d39/Base_epoch8_ckpt.pt")


In [ ]:
#@title **Get WRAPs git repo**
%%time
import os, time
if not os.path.isdir("WRAPs"):
  run_cmd("git clone https://github.com/davidekim/WRAPs.git")

In [ ]:
#@title **Create secondary structure and block adjacency inputs for C8 symmetric outputs**

#specify helix and loop length and connections
helix_length = 21
loop_length = 4
#specify number of helices and loops in ss_features input
no_helix = 2
no_loop = 1

features = f"1 H {helix_length}\n\
2 L {loop_length}\n\
3 H {helix_length}\n\
Intra 1,3\n\
Inter 3,1"

ss_folder = f'C8'
if not os.path.isdir(ss_folder):
  os.system(f"mkdir {ss_folder}")

calc_contig_length = (no_helix * helix_length) + (no_loop * loop_length)
print("calc_contig_length = ", calc_contig_length)
outname = ss_folder + '/ss_features.txt'
with open(outname, 'w') as f:
    f.writelines(features)
run_cmd(f'python WRAPs/helper_scripts/symm_make_ss_adj.py --def_file {outname} --nsub 8 --outfolder {ss_folder}')
from IPython.display import Image
Image(f'{ss_folder}/adj.png', width=600)

In [ ]:
#@title **Run symmetric RFdiffusion** (~10min to a long time depending on num_designs)
num_designs = 1 #@param ["1", "5", "10", "50"] {type:"raw"}
contig_length = calc_contig_length*8

cmd = f'{diffusion_script} contigmap.contigs=[\"{contig_length}-{contig_length}\"] inference.symmetry="C8" '
cmd += f"inference.output_prefix='C8_output/C8' diffuser.T=50 "
cmd += f"inference.num_designs={num_designs} denoiser.noise_scale_ca=0.5 "
cmd += f"denoiser.noise_scale_frame=0.5 scaffoldguided.scaffoldguided=True scaffoldguided.scaffold_dir={ss_folder} "

print(cmd)
run_cmd(cmd)

import glob
symm_wraps = []
for output in glob.glob('C8_output/C8_*.pdb'):
  #convert to single chain
  tmpstr = ""
  with open(output) as f:
    for l in f:
      if l.startswith("ATOM"):
        tmpstr += l[0:20]+" A"+l[22:]
  with open(output, "w") as out: out.write(tmpstr)
  symm_wraps.append(output)

In [ ]:
#@title **Select symmetric output to connect into a single chain**
import py3Dmol
import ipywidgets as widgets
from ipywidgets import interact, Layout
from IPython.display import display, clear_output

current_symm = ""
dropdown = widgets.Dropdown(
  options=symm_wraps,
  description='symm:',
  layout=Layout(width='50%', overflow='visible')
)
def on_dropdown_change(wrap):
  global current_symm
  current_symm = wrap
  clear_output(wait=True)
  view = py3Dmol.view(width=500, height=400)
  with open(wrap, "r") as f:
    pdb_data = f.read()
  view.addModel(pdb_data, 'pdb')
  view.setStyle({'cartoon': {'color':'magenta'}})
  view.zoomTo()
  view.show()

widgets.interact(on_dropdown_change, wrap=dropdown);

print(current_symm)

In [ ]:
#@title **Connect symmetric diffusion output(s) with loops into single chains** (~5min to a long time)
num_designs = 1 #@param ["1", "5", "10", "50"] {type:"raw"}

contigs_OmpA =  "A6-46/3-3/A52-92/3-3/A98-138/3-3/A144-184/3-3/A190-230/3-3/A236-276/3-3/A282-322/3-3/A329-368"
contigs_GlpG = "A1-46/15-15/A52-92/10-10/A93-138/10-10/A139-184/10-10/A185-230/10-10/A231-276/10-10/A278-322/10-10/A323-368/10-10"

contigs = { 'select contigs': ''}
contigs[f'GlpG: {contigs_GlpG}'] = contigs_GlpG
contigs[f'OmpA and TPs: {contigs_OmpA}'] = contigs_OmpA

dropdown = widgets.Dropdown(
  options=contigs,
  description='contigs:',
  layout=Layout(width='50%', overflow='visible')
)
contigs_str = ""
def on_dropdown_change(contigs):
  contigs_str = contigs
  if len(contigs_str) > 0:
    cmd = f"{diffusion_script} inference.output_prefix=h16_output/h16 "
    cmd += f"inference.input_pdb={current_symm} diffuser.T=30 "
    cmd += f'contigmap.contigs=["{contigs_str}"] '
    cmd += f"inference.num_designs={num_designs} denoiser.noise_scale_ca=0.5 denoiser.noise_scale_frame=0.5"

    # clear previous outputs
    os.system("rm -rf h16_output")
    print(cmd)
    run_cmd(cmd)

widgets.interact(on_dropdown_change, contigs=dropdown);

import glob
wraps = []
for output in glob.glob('h16_output/h16_*.pdb'):
  wraps.append(output)


In [ ]:
#@title **Select wrap to use**
import py3Dmol
import ipywidgets as widgets
from ipywidgets import interact, Layout
from IPython.display import display, clear_output

current_wrap = ""
dropdown = widgets.Dropdown(
  options=wraps,
  description='wrap:',
  layout=Layout(width='50%', overflow='visible')
)
def on_dropdown_change(wrap):
  global current_wrap
  current_wrap = wrap
  clear_output(wait=True)
  view = py3Dmol.view(width=500, height=400)
  with open(wrap, "r") as f:
    pdb_data = f.read()
  view.addModel(pdb_data, 'pdb')
  view.setStyle({'cartoon': {'color':'magenta'}})
  view.zoomTo()
  view.show()

widgets.interact(on_dropdown_change, wrap=dropdown);

In [ ]:
#@title **Get sushimaki to place wrap around target** (~5min)

if not os.path.isdir("sushimaki"):
  print("installing sushimaki...")
  os.system("git clone https://github.com/davidekim/sushimaki.git")
  os.system("cd sushimaki; git submodule init; git submodule update --remote;")
  os.system("cd sushimaki/ppi_iterative_opt/rf_diffusion/env/SE3Transformer; pip install .")

  # install DeepTMHMM
  os.system("pip3 install --upgrade pybiolib")
  # install PyRosetta
  os.system("pip install pyrosetta --find-links https://west.rosettacommons.org/pyrosetta/quarterly/release")

print()
run_cmd("python sushimaki/sushimaki.py")

In [ ]:
%%time
import glob
from google.colab import files

#@title **Place wrap around target PDB using sushimaki** (~5min)
#@markdown PDB must be a single chain and transmembrane protein
target_to_wrap = "2ge4A OmpA" #@param ["2ge4A OmpA","2ic8A GlpG", "TP0733", "TP0126", "TP0698", "Upload your own transmembrane target PDB"] {type: "string"}

# clear previous inputs
os.system("rm -rf input_DeepTMHMM input.pdb input_WRAP*.pdb partial_diffusion_task_file_input*.txt")
input_pdb_str = ""
target_to_wrap = target_to_wrap.split()[0]
if target_to_wrap.startswith("TP"):
  os.system(f"cp WRAPs/PB*_{target_to_wrap}/{target_to_wrap}.pdb input.pdb")
  input_pdb_str = target_to_wrap
elif not target_to_wrap.startswith("Upload") and len(target_to_wrap) > 4:
  pdb_code = target_to_wrap[0:4]
  pdb_chain = target_to_wrap[4:]
  if not os.path.isfile(f"{pdb_code}.pdb1"):
    os.system(f"wget -qnc https://files.rcsb.org/download/{pdb_code}.pdb1.gz")
    os.system(f"gunzip {pdb_code}.pdb1.gz")
  with open(f"{pdb_code}.pdb1") as f:
    for l in f:
      if l.startswith("ATOM") and l[20:22].strip() == pdb_chain:
        input_pdb_str += l
  with open("input.pdb", "w") as out: out.write(input_pdb_str)
else:
  uploads = files.upload()
  if uploads:
    target_to_wrap = list(uploads.keys())[0].split('.pdb')[0].split('/')[-1].split()[0]
    input_pdb_str = uploads[list(uploads.keys())[0]]
    with open("input.pdb", "wb") as out: out.write(input_pdb_str)

if len(input_pdb_str) > 0:
  print()
  print(target_to_wrap)
  print()
  cmd = f"python sushimaki/sushimaki.py --wrap {current_wrap} input.pdb"
  print(cmd)
  run_cmd(cmd)

target_wraps = []
for wrap in glob.glob('input_WRAP*.pdb'):
  target_wraps.append(wrap)

In [ ]:
#@title **Select wrapped target to use**

current_wrapped_target = ""
dropdown = widgets.Dropdown(
  options=target_wraps,
  description='wrap:',
  layout=Layout(width='50%', overflow='visible')
)
def on_dropdown_change(wrap):
  global current_wrapped_target
  current_wrapped_target = wrap
  clear_output(wait=True)
  print()
  print(current_wrapped_target)
  view = py3Dmol.view(width=500, height=400)
  with open(wrap, "r") as f:
    pdb_data = f.read()
  view.addModel(pdb_data, 'pdb')
  view.setStyle({'cartoon': {'color':'magenta'}})
  view.zoomTo()
  view.show()

widgets.interact(on_dropdown_change, wrap=dropdown);



In [ ]:
#@title **Download ppi_iterative_opt RFdiffusion checkpoint**
%%time
if not os.path.exists("sushimaki/ppi_iterative_opt/rf_diffusion/models/BFF_4.pt"):
  os.system("mkdir sushimaki/ppi_iterative_opt/rf_diffusion/models")
  os.system("cd sushimaki/ppi_iterative_opt/rf_diffusion/models; wget https://files.ipd.uw.edu/pub/ppi_iterative_opt/rf_diffusion/models/BFF_4.pt")

In [ ]:
#@title **Download AF2 params** (~5min)
%%time
if not os.path.isdir("sushimaki/ppi_iterative_opt/af2_initial_guess/params"):
  os.system("mkdir sushimaki/ppi_iterative_opt/af2_initial_guess/params")
  os.system("cd sushimaki/ppi_iterative_opt/af2_initial_guess/params; wget https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; tar -xf alphafold_params_2022-12-06.tar")

In [ ]:
#@title Run **ppi_iterative_opt** partial diffusion -> mpnn -> af2 optimization on selected wrap.
#@markdown Runtime depends on partial_diffusions, total_traj, and cycles (~1 to many hours).

#@markdown Backbone diversity increases with partial_T.

sushimaki_wrap = current_wrapped_target
partial_T = 20 #@param ["10", "15", "20", "25", "30"] {type:"raw"}
partial_diffusions = 5 #@param ["1", "5", "10"] {type:"raw"}
total_traj = 1 #@param ["1", "2", "3", "4", "5"] {type:"raw"}
cycles = 1 #@param ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"] {type:"raw"}

# clear previous ppi_iterative_opt output
os.system("rm -rf ppi_iterative_opt_output; rm af2.sc; rm check.point_*;")

cmd = f"python sushimaki/ppi_iterative_opt/ppi_iterative_opt.py --partial_T {partial_T} --partial_diffusions {partial_diffusions} --cycles {cycles} --total_traj {total_traj} {sushimaki_wrap}"
print(cmd)
run_cmd(cmd)

In [ ]:
#@title **Plot AF2 plddt_binder vs pae_interaction of ppi_iterative_opt wraps**
import pandas as pd
import matplotlib.pyplot as plt
os.system("cat ppi_iterative_opt_output/*_af2.sc | head -n 1 > af2.sc")
os.system("cat ppi_iterative_opt_output/*_af2.sc | grep -v plddt_total | sort -n -k3 >> af2.sc")
df_af2 = pd.read_csv('af2.sc', sep=r'\s+')

def scatter_hist(x, y, ax, ax_histx, ax_histy, color, xlabel, ylabel):
    # no labels
    ax_histx.tick_params(axis="x", labelbottom=False)
    ax_histy.tick_params(axis="y", labelleft=False)

    # the scatter plot:
    ax.scatter(x, y)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax_histx.hist(x, bins=100)
    ax_histy.hist(y, orientation='horizontal', bins=100)

# Start with a square Figure.
fig = plt.figure(figsize=(8, 4))
gs = fig.add_gridspec(2, 2,  width_ratios=(4, 1), height_ratios=(1, 4),
                      left=0.1, right=0.9, bottom=0.1, top=0.9,
                      wspace=0.05, hspace=0.05)
# Create the Axes.
ax = fig.add_subplot(gs[1, 0])
ax_histx = fig.add_subplot(gs[0, 0], sharex=ax)
ax_histy = fig.add_subplot(gs[1, 1], sharey=ax)
# Draw the scatter plot and marginals.
scatter_hist(df_af2['pae_interaction'],df_af2['plddt_binder'], ax, ax_histx, ax_histy, 'black', 'pae interaction', 'plddt binder')


In [ ]:
#@title **Download AF2 WRAP**
af2_wraps = { 'Select to download': ''}
for i,r in df_af2.iterrows():
  name = target_to_wrap + '_' + r['description'].split('/')[-1]+'.pdb'
  af2_wraps[f"{name} pae_i: {r['pae_interaction']} plddt_binder: {r['plddt_binder']}"] = r['description']+'.pdb'

dropdown = widgets.Dropdown(
  options=af2_wraps,
  description='af2 wrap:',
  layout=Layout(width='50%', overflow='visible')
)
def on_dropdown_change(af2_wrap):
  if os.path.exists(af2_wrap):
    name = target_to_wrap +'_' + af2_wrap.split('/')[-1]
    os.system(f"cp {af2_wrap} {name}")
    files.download(name)

widgets.interact(on_dropdown_change, af2_wrap=dropdown);